# RNA 3D Structure Prediction — Kaggle GPU Training

Trains the full MSA Transformer + Geometric Structure Module on **all 5 716 training sequences**.

**Before running:**
1. Enable GPU in *Notebook settings → Accelerator → GPU T4 x2* (or P100)
2. Add the competition dataset: *+ Add Data → `stanford-rna-3d-folding`*
3. (Optional) Make the notebook public or use a private fork if the GitHub repo is private.

Estimated runtime on T4: ~6–8 h for 30 epochs on the full training set.

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device  : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {mem:.1f} GB')
else:
    print('WARNING: No GPU detected — training will be very slow.')

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'biopython>=1.81', 'einops>=0.6.1', 'tqdm>=4.65.0'],
    check=True
)
print('Dependencies ready.')

In [ ]:
import os, subprocess

REPO   = 'https://github.com/shanmugavalli/rna-structure.git'
WORKDIR = '/kaggle/working/rna-structure'

if os.path.exists(f'{WORKDIR}/.git'):
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
print(f'Working directory: {os.getcwd()}')

for f in ['src/train.py', 'src/config.py', 'src/model.py', 'src/dataset.py']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}] {f}')

In [ ]:
import os

# Kaggle places competition data here
COMPETITION_INPUT = '/kaggle/input/stanford-rna-3d-folding'

# Create output directories
for d in ['data/raw', 'data/processed', 'outputs/checkpoints',
          'outputs/logs', 'outputs/predictions', 'outputs/analysis']:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(COMPETITION_INPUT):
    print(f'Competition input not at {COMPETITION_INPUT}.')
    print('Available inputs:')
    for d in os.listdir('/kaggle/input/'):
        print(f'  /kaggle/input/{d}/')
    raise FileNotFoundError(
        'Update COMPETITION_INPUT above to the correct path.')

print(f'Competition data found at {COMPETITION_INPUT}')

# Symlink CSV files so the training code finds them at expected paths
csv_files = [
    'train_sequences.csv', 'train_labels.csv',
    'validation_sequences.csv', 'validation_labels.csv',
    'test_sequences.csv', 'sample_submission.csv',
]
for fname in csv_files:
    src = os.path.join(COMPETITION_INPUT, fname)
    dst = os.path.join('data/raw', fname)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'  linked: {fname}')
    elif not os.path.exists(src):
        print(f'  WARNING: {fname} not found in competition input')

# Symlink MSA directory
msa_src = os.path.join(COMPETITION_INPUT, 'MSA')
msa_dst = 'data/raw/MSA'
if os.path.exists(msa_src):
    if not os.path.lexists(msa_dst):
        os.symlink(msa_src, msa_dst)
        n_msa = len(os.listdir(msa_src))
        print(f'  linked: MSA/ ({n_msa} files)')
else:
    print('  WARNING: MSA/ folder not found — use_msa will be disabled')
    # Make sure use_msa won't crash
    os.makedirs(msa_dst, exist_ok=True)

print('Data setup complete.')

## Training Configuration

| Setting | Value | Notes |
|---|---|---|
| `RNA_RUNTIME` | `gpu` | selects `full` model + CUDA |
| `RNA_EPOCHS` | `30` | ~6–8 h on T4 |
| `RNA_LR` | `1e-4` | GPU default |
| `RNA_BATCH_SIZE` | `1` | + grad_accum=8 → effective 8 |
| `RNA_MAX_SEQ_LENGTH` | `256` | safe for 16 GB VRAM |
| `RNA_MAX_MSA_SEQS` | `16` | more MSA than default(6) |
| `RNA_MSA_DEPTH` | `4` | 4 Evoformer blocks |
| `RNA_STRUCTURE_ITERATIONS` | `3` | 3 refinement cycles |
| `RNA_AUG_ROT` | `0.8` | rotation augmentation |
| `RNA_AUG_NOISE` | `0.3` | coordinate noise |

Tweak `extra_env` in the next cell if you want to change any setting.

In [ ]:
import os, subprocess, sys

extra_env = {
    'RNA_RUNTIME'              : 'gpu',
    'RNA_EPOCHS'               : '30',
    'RNA_LR'                   : '1e-4',
    'RNA_BATCH_SIZE'           : '1',
    'RNA_MAX_SEQ_LENGTH'       : '256',
    'RNA_MAX_MSA_SEQS'         : '16',
    'RNA_MSA_DEPTH'            : '4',
    'RNA_STRUCTURE_ITERATIONS' : '3',
    'RNA_AUG_ROT'              : '0.8',
    'RNA_AUG_NOISE'            : '0.3',
    'RNA_AUG_NOISE_STD'        : '0.1',
    'RNA_VAL_FREQUENCY'        : '1',
}

env = {**os.environ, **extra_env}
LOG = 'outputs/logs/kaggle_gpu_train.log'

print('Starting GPU training...')
print('Config overrides:', extra_env)
print('-' * 70)

with open(LOG, 'w') as log_f:
    proc = subprocess.Popen(
        [sys.executable, '-u', 'src/train.py'],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
    proc.wait()

print(f'\nExit code : {proc.returncode}')
print(f'Log saved : {LOG}')

In [ ]:
import re, glob, os

LOG = 'outputs/logs/kaggle_gpu_train.log'

# --- Parse TM scores from log ---
tm_history = []
with open(LOG) as f:
    for line in f:
        m = re.search(r'\[Epoch\s+(\d+)\].*Val TM:\s+([0-9.]+)', line)
        if not m:
            m = re.search(r'Val TM:\s+([0-9.]+)', line)
            if m:
                tm_history.append(float(m.group(1)))
        else:
            tm_history.append(float(m.group(2)))

if tm_history:
    best = max(tm_history)
    last = tm_history[-1]
    print(f'Epochs recorded : {len(tm_history)}')
    print(f'Best Val TM     : {best:.4f}')
    print(f'Last Val TM     : {last:.4f}')
else:
    print('No Val TM lines found in log — check log for errors.')

# --- Last 30 lines of log ---
print('\n--- Last 30 log lines ---')
with open(LOG) as f:
    tail = f.readlines()[-30:]
print(''.join(tail))

# --- List checkpoints ---
ckpts = sorted(glob.glob('outputs/checkpoints/*.pt'))
print(f'\nCheckpoints saved ({len(ckpts)}):')
for c in ckpts:
    print(f'  {os.path.basename(c)}  ({os.path.getsize(c)/1e6:.1f} MB)')

In [ ]:
import shutil, os

# Zip log + checkpoints for easy download from Kaggle outputs panel
shutil.make_archive('/kaggle/working/training_results', 'zip', 'outputs')
size = os.path.getsize('/kaggle/working/training_results.zip') / 1e6
print(f'training_results.zip created: {size:.1f} MB')
print('Download from the Kaggle output panel on the right.')